# Demonstration Notebook: Spectrum 1D

This notebook demonstrates the config-free ('deconfigged') API when looking at images in the Notebook setting. This notebook follows the examples of the SpecvizExample notebook and refers to the plugins used by Specviz. UI equivalents for these actions, as well as additional documentation about the Specviz configuration generally, can be found here: https://jdaviz.readthedocs.io/en/latest/specviz/

Import modules needed for this notebook.

In [ ]:
import warnings
import jdaviz

## Launch App

We create a Jdaviz app ('deconfigged') instance.

In [ ]:
# gca() refers to 'get current app'. It is not strictly necessary here but can be useful
# if multiple instances of the app are instantiated.
jd = jdaviz.gca()

## Display App

This will show the app inline in the notebook. You can pass an integer to the `height` keyword to change the displayed height of the app (the default is 600). You can also pass `loc="popout:window"` to immediately pop out the Jdaviz app into a separate window.

In [ ]:
jd.show()

## Load Data into the App

Now we load observations. If you already have the files on your local machine, you can 
load them by passing their file paths to `load` as strings. For this example, 
we will retrieve remote data from [MAST](https://mast.stsci.edu/) via `astroquery`
by passing the observation's URI as a string. By default, the downloaded files are 
saved to the current working directory. If you set `cache=False` in the `load` 
call, it will re-download the file if desired.

One other thing to note about retrieving MAST data through astroquery is that it caches
the data by default. It is possible for files to be updated in MAST with more recent calibrations
but remain the same size, in which case your local cached file would not be replaced by the new
version. Again, you can turn off caching by setting `cache=False` to force it to re-download the 
file.

In [ ]:
uri = 'mast:JWST/product/jw02732-c1001_t004_miri_ch1-short_x1d.fits'

data_label = 'my_spectrum'
# Suppress warnings
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    jd.load(uri, format='1D Spectrum', data_label=data_label, cache=True)

Alternatively, the data format can be autodetected and the data loaded simultaneously by calling `import jdaviz; jdaviz.open(uri or filename)`

## Retrieve Data and Subsets (and sometimes both!)

User Task: Select a subset in the viewer above

After defining regions in the spectra, you can retrieve your data with the user-defined subset mask:

In [ ]:
# Returns the 'raw' data used by the app
my_spectrum_data = jd.datasets[data_label].get_data()

# Returns the 'raw' data used by the app with a mask applied 
# to represent the subset
my_spectrum_data_masked = jd.datasets[data_label].get_data(spectral_subset='Subset 1')
# Extract the mask from the returned specutils.Spectrum object
my_subset = my_spectrum_data_masked.mask
# Apply the mask to the spectral axis to produce the 'masked region'
masked_region = my_spectrum_data_masked.spectral_axis[~my_subset]
print(f'Your subset spans from {masked_region[0]:.4f} to {masked_region[-1]:.4f}.')

And to retrieve the subset from the **Subset Tools** plugin as a `specutils.SpectralRegion` object,

In [ ]:
# NOTE: The bounds of the spectral region won't exactly match the bounds
# of the masked region due to...
spectral_region = jd.plugins['Subset Tools'].get_regions()['Subset 1']
print(f'Your subset spans from approximately {spectral_region.bounds[0]:.4f} to {spectral_region.bounds[-1]:.4f}.')

## Export Viewer Image

This can be done in the API as follows or by clicking on the **Export** icon (<!-- ../jdaviz/data/icons/content-save.svg --><img src="data:image/svg+xml;utf8,<svg%20xmlns%3D%22http://www.w3.org/2000/svg%22%20viewBox%3D%220%200%2024%2024%22><path%20d%3D%22M15%2C9H5V5H15M12%2C19A3%2C3%200%200%2C1%209%2C16A3%2C3%200%200%2C1%2012%2C13A3%2C3%200%200%2C1%2015%2C16A3%2C3%200%200%2C1%2012%2C19M17%2C3H5C3.89%2C3%203%2C3.9%203%2C5V19A2%2C2%200%200%2C0%205%2C21H19A2%2C2%200%200%2C0%2021%2C19V7L17%2C3Z%22%20fill%3D%22%23000000%22%20stroke%3D%22%23ffffff%22%20stroke-width%3D%221.75%22%20stroke-linecap%3D%22round%22%20stroke-linejoin%3D%22round%22%20vector-effect%3D%22non-scaling-stroke%22%20style%3D%22paint-order:%20stroke%20fill%22%20/></svg>" width="16" style="vertical-align:-4px;" />) and proceeding from there.

In [ ]:
exp = jd.plugins['Export']
exp.viewer = '1D Spectrum'
exp.filename = f'{data_label}.png'
print(f'Exported to: {exp.export(overwrite=True)}')

## Pan/Zoom in Specviz
You can use modify the field of view of the spectrum viewer by setting its limits. You can provide it a scalar (which assumes the units of the loaded spectra) or an `astropy.Quantity`.

In [ ]:
viewer = jd.viewers['1D Spectrum']
viewer.set_limits(x_min=7, x_max=7.2, y_max=170.0)

### Disable Scientific Notation
Scientific notation is used in the axes of the viewers by default. To deactivate it, run the following code:

In [ ]:
# fmt can be '0.1e' to set scientific notation or '0.2f' to turn it off
viewer.set_tick_format(fmt='0.2f', axis='x')

### Reset Zoom
You can also quickly return to the default zoom,

In [ ]:
viewer.reset_limits()

## Extract Models
If models were added using the model fitting plugin, they can be extracted using the following property,

In [ ]:
model_fitting = jd.plugins['Model Fitting']
model_fitting.fitted_models

If the name of a particular model is known, it can be extracted from the `fitted_models` property,

In [ ]:
model_name = 'model'
model_fitting.fitted_models.get(model_name, f"No models named '{model_name}' have been fit to the data.")